# CropSense AI: Model Training, Evaluation & Explainability
Covers:
- Tabular Random Forest & XGBoost risk modeling
- MobileNetV2 transfer learning with calibrated temperature scaling
- Classical SVM benchmark comparison on OpenCV features
- Grad-CAM visual activation maps

In [ ]:
import torch
import json
import pandas as pd
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from src.train_cnn import CropSenseCNN, get_image_transforms, SUPPORTED_CLASSES
from src.gradcam import GradCAM, overlay_gradcam

with open('../reports/tabular_evaluation.json') as f:
    tab_eval = json.load(f)
print("Tabular RF Accuracy:", tab_eval['rf_metrics']['accuracy'])
print("Tabular XGBoost Accuracy:", tab_eval['xgb_metrics']['accuracy'])
print("Top Features:", tab_eval['feature_importances'])

## 1. Grad-CAM Explainability Demonstration

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CropSenseCNN(num_classes=len(SUPPORTED_CLASSES), pretrained=False)
model.load_state_dict(torch.load('../models/cropsense_cnn.pth', map_location=device))
model.to(device)
model.eval()

gradcam = GradCAM(model, model.features[-1])
img_bgr = cv2.imread('../data/raw/images/leaf_0000_Early_Blight.png')
if img_bgr is None:
    img_bgr = cv2.imread(glob.glob('../data/raw/images/*.png')[0])

rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
pil_img = Image.fromarray(rgb)
_, eval_tf = get_image_transforms()
tensor = eval_tf(pil_img).unsqueeze(0).to(device)

probs, _ = model.predict_calibrated(tensor)
pred_idx = torch.argmax(probs, dim=1).item()
print(f"Predicted: {SUPPORTED_CLASSES[pred_idx]} with confidence {probs[0, pred_idx]:.3f}")

cam = gradcam.generate_cam(tensor, target_class=pred_idx)
overlay = overlay_gradcam(img_bgr, cam, alpha=0.55)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.title('Original Leaf'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); plt.title('Grad-CAM Heatmap'); plt.axis('off')
plt.tight_layout()
plt.show()